# Sheet 1.1: Core AI intuitions, in PyTorch and TensorFlow

**Roadmap topic 1.1** (Fanout module 02). Read first: PyTorch broadcasting semantics, the NumPy broadcasting guide, the TF tensor guide, *The Illustrated Word2vec*.

Every exercise asks for a **PyTorch and a TensorFlow version**. Keep notes on what felt different; you will need them for the framework memo in sheet 1.2.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells, no Python loops over batch elements unless a task allows it. CPU is fine.

In [ ]:
import math, numpy as np, torch
import torch._dynamo  # load before TensorFlow: avoids an LLVM symbol clash in some torch+TF builds
import tensorflow as tf
torch.manual_seed(0); tf.random.set_seed(0)

## Exercise 1: Predict broadcasting before you run it

For each pair of shapes, write the shape of `a + b` as a tuple, or the string `"error"` if it fails. Do it **in your head** first; the check tells you which ones you got wrong but not the answers.

In [ ]:
pairs = [((3, 1), (4,)), ((2, 3), (3, 2)), ((5, 1, 4), (3, 1)), ((1,), (7, 0)), ((8, 1, 6, 1), (7, 1, 5)),
         ((4, 3), (4,)), ((2, 1, 3), (2, 3)), ((6,), (6, 1)), ((1, 1), ()), ((3, 4, 5), (4, 1, 5))]
predictions = [None] * 10   # YOUR ANSWERS, e.g. (3, 4) or "error"

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    wrong = []
    for i, ((sa, sb), p) in enumerate(zip(pairs, predictions)):
        try: true = tuple(torch.zeros(sa) .add(torch.zeros(sb)).shape)
        except RuntimeError: true = "error"
        try: tft = tuple((tf.zeros(sa) + tf.zeros(sb)).shape)
        except Exception: tft = "error"
        assert true == tft
        if p != true: wrong.append(i)
    assert not wrong, f"wrong predictions at indices {wrong}; work them out again with the right-to-left rule"
    print("PASS  10/10")
_check1()

## Exercise 2: Pairwise distances at scale

Retrieval, k-NN, k-means and contrastive losses all need the matrix of squared L2 distances between two sets of vectors.

Implement `pdist2_torch(A, B, chunk)` and `pdist2_tf(A, B, chunk)` using `‖a‖² + ‖b‖² - 2abᵀ`:
- no Python loop over rows, except a loop over **chunks** of `A` of size `chunk` to bound memory
- the result must never be negative (floating-point cancellation can make it slightly negative; handle it)
- float32 inputs, float32 output

In [ ]:
def pdist2_torch(A, B, chunk=4096):
    # YOUR CODE
    raise NotImplementedError
def pdist2_tf(A, B, chunk=4096):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    A = torch.randn(3000, 64) * 10 + 100; B = torch.cat([A[:500] + 1e-3 * torch.randn(500, 64), torch.randn(700, 64)])
    ref = torch.cdist(A.double(), B.double()) ** 2
    d = pdist2_torch(A, B, chunk=777)
    assert d.dtype == torch.float32 and d.shape == (3000, 1200)
    assert (d >= 0).all(), "negative squared distances"
    assert torch.allclose(d.double(), ref, rtol=1e-3, atol=2.0), "torch distances wrong"
    dt = pdist2_tf(tf.constant(A.numpy()), tf.constant(B.numpy()), chunk=777).numpy()
    assert (dt >= 0).all() and np.allclose(dt, ref.numpy(), rtol=1e-3, atol=2.0), "tf distances wrong"
    print("PASS")
_check2()

**Written (2a):** With inputs around 100 and near-duplicate pairs, how large is the cancellation error of `‖a‖²+‖b‖²-2ab` in float32 compared with the true distance? When would you compute `‖a-b‖²` directly instead?

**Your answer:**

## Exercise 3: Softmax that survives fp16

Implement `softmax`, `log_softmax` for the last axis in both frameworks **in the input's dtype** (float16 here). They must produce no `inf`/`nan` for logits up to ±6e4 (near the fp16 max) and match a float64 reference to fp16 precision.

In [ ]:
def softmax_torch(x): raise NotImplementedError
def log_softmax_torch(x): raise NotImplementedError
def softmax_tf(x): raise NotImplementedError
def log_softmax_tf(x): raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    x64 = torch.cat([torch.randn(8, 512) * 3, torch.tensor([[6e4, -6e4] + [0.] * 510])]).double()
    x16 = x64.half(); ref_p = torch.softmax(x16.double(), -1); ref_l = torch.log_softmax(x16.double(), -1)
    for name, sm, lsm, conv in (("torch", softmax_torch, log_softmax_torch, lambda t: t),
                                ("tf", softmax_tf, log_softmax_tf, lambda t: tf.constant(t.numpy()))):
        p = sm(conv(x16)); l = lsm(conv(x16))
        p = torch.as_tensor(np.asarray(p)); l = torch.as_tensor(np.asarray(l))
        assert p.dtype == torch.float16 and l.dtype == torch.float16, f"{name}: keep fp16"
        assert torch.isfinite(p).all(), f"{name}: softmax has inf/nan"
        assert torch.isfinite(l[:8]).all(), f"{name}: log_softmax has inf/nan on normal rows"
        assert torch.allclose(p.double(), ref_p, atol=2e-3), f"{name}: softmax inaccurate"
        mask = ref_l > -1e4
        assert torch.allclose(l.double()[mask], ref_l[mask], atol=2e-2, rtol=2e-3), f"{name}: log_softmax inaccurate"
    print("PASS")
_check3()

## Exercise 4: Dot, cosine and L2 ranking

1. `rank(q, D, metric)` returns document indices sorted best-first for `metric` in `{"dot", "cos", "l2"}` (PyTorch only).
2. Prove they agree for unit vectors (the check tests it).
3. `counterexample()` returns `(q, D)` with 2-D vectors where **dot product picks a different top-1 than cosine** because one document vector is long. This is the failure you get when a search index stores unnormalised embeddings.

In [ ]:
def rank(q, D, metric):
    # YOUR CODE
    raise NotImplementedError
def counterexample():
    # YOUR CODE: return q (2,), D (n, 2)
    raise NotImplementedError

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    D = torch.nn.functional.normalize(torch.randn(500, 32), dim=1); q = torch.nn.functional.normalize(torch.randn(32), dim=0)
    r = [rank(q, D, m) for m in ("dot", "cos", "l2")]
    assert torch.equal(r[0], r[1]) and torch.equal(r[1], r[2]), "for unit vectors all three rankings must agree"
    q, D = counterexample()
    assert q.shape == (2,) and D.shape[1] == 2
    assert rank(q, D, "dot")[0] != rank(q, D, "cos")[0], "dot and cosine pick the same top-1 here"
    print("PASS")
_check4()

### On Colab: semantic search on real embeddings
`pip install sentence-transformers`, embed 10,000 sentences from any dataset (e.g. `ag_news` via `datasets`) with `all-MiniLM-L6-v2`, then run 20 queries with each metric, with and without normalisation. Report how often the top-10 sets differ and show one query where unnormalised dot product returns a long, off-topic sentence.

In [ ]:
# Colab: your semantic search experiment

## Exercise 5: einsum in both frameworks

Implement each with a **single** `einsum` call in PyTorch and in TF:
- `attn_scores(Q, K)`: `Q (B, H, T, d)`, `K (B, H, S, d)` → `(B, H, T, S)` scores (no scaling)
- `attn_apply(P, V)`: `P (B, H, T, S)`, `V (B, H, S, d)` → `(B, H, T, d)`
- `bilinear(x, W, y)`: `x (B, i)`, `W (o, i, j)`, `y (B, j)` → `(B, o)`, as in `torch.nn.Bilinear` without bias
- `batch_trace_AB(A, B)`: `A (N, n, m)`, `B (N, m, n)` → `trace(A_k B_k)` for each k, **without** forming the product

In [ ]:
def attn_scores_t(Q, K): raise NotImplementedError
def attn_apply_t(P, V): raise NotImplementedError
def bilinear_t(x, W, y): raise NotImplementedError
def batch_trace_AB_t(A, B): raise NotImplementedError
def attn_scores_tf(Q, K): raise NotImplementedError
def attn_apply_tf(P, V): raise NotImplementedError
def bilinear_tf(x, W, y): raise NotImplementedError
def batch_trace_AB_tf(A, B): raise NotImplementedError

In [ ]:
# ---- CHECK 5 (do not edit) ----
def _check5():
    Q, K, V = torch.randn(2, 3, 5, 4), torch.randn(2, 3, 7, 4), torch.randn(2, 3, 7, 4); P = torch.softmax(torch.randn(2, 3, 5, 7), -1)
    x, W, y = torch.randn(6, 4), torch.randn(3, 4, 5), torch.randn(6, 5)
    A, B = torch.randn(9, 4, 6), torch.randn(9, 6, 4)
    ref = [Q @ K.transpose(-1, -2), P @ V, torch.nn.functional.bilinear(x, y, W), torch.diagonal(A @ B, dim1=1, dim2=2).sum(1)]
    mine_t = [attn_scores_t(Q, K), attn_apply_t(P, V), bilinear_t(x, W, y), batch_trace_AB_t(A, B)]
    tfc = lambda t: tf.constant(t.numpy())
    mine_tf = [attn_scores_tf(tfc(Q), tfc(K)), attn_apply_tf(tfc(P), tfc(V)), bilinear_tf(tfc(x), tfc(W), tfc(y)), batch_trace_AB_tf(tfc(A), tfc(B))]
    for i, (r, a, b) in enumerate(zip(ref, mine_t, mine_tf)):
        assert torch.allclose(a, r, atol=1e-4), f"torch op {i} wrong"
        assert np.allclose(b.numpy(), r.numpy(), atol=1e-4), f"tf op {i} wrong"
    print("PASS")
_check5()

## Checkpoint

Without running code, predict the output shape or the error for 10 random shape pairs a friend gives you (or generate them with `torch.randint`). 10/10 required. Also explain in one sentence each: why softmax subtracts the max, and why cosine similarity is the default for embedding search.

---
### Hints for self-grading
- 2a: the absolute error is about `eps32 · ‖a‖² ≈ 1e-7 · 6.4e5 ≈ 0.06`, while near-duplicate distances are about `64·1e-6`, so relative error is huge. Compute `‖a-b‖²` directly (or in float64) when you care about near neighbours.